### 1. nn.Embedding

ref - https://docs.pytorch.org/docs/2.12/generated/torch.nn.Embedding.html

nn.Embedding is a lookup table that maps integer indices to dense vectors.

embedding = nn.Embedding(num_embeddings, embedding_dim)

num_embeddings: vocabulary size (number of unique tokens)

embedding_dim: size of each embedding vector

Internally, it holds a weight matrix of shape (num_embeddings, embedding_dim). Given an integer index (or batch of indices), it simply indexes into that matrix — no matrix multiplication, just a lookup.

So the embedding layer itself has no concept of English words — it just knows indices. The mapping from words→indices is handled externally (e.g. a vocabulary dictionary, BPE tokenizer like GPT's, etc.), and indices to actual vectors could be done by nn.Embedding.

In [1]:
import torch
import torch.nn as nn

/Users/suryamgupta/Documents/AI_Projects/transformers-from-scratch/.venv/lib/python3.11/site-packages/torch/_subclasses/functional_tensor.py:362: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [2]:
embedding = nn.Embedding(10, 4)  # 10 tokens, 4-dim vectors

idx = torch.tensor([2, 5, 2])
out = embedding(idx)  # shape: (3, 4)

In [3]:
out

tensor([[ 1.1974, -0.1416,  1.7437,  0.0953],
        [ 1.7315, -1.2789, -0.2950, -0.6234],
        [ 1.1974, -0.1416,  1.7437,  0.0953]], grad_fn=<EmbeddingBackward0>)

In [4]:
out[0] == embedding.weight[2]

tensor([True, True, True, True])

### 2. In code of positional encoding class

In [11]:
import math
seq_len = 10 # lets say a sentence has 10 tokens
d_model = 16

In [10]:
# position = torch.arange(0,seq_len).unsqueeze(1)  # shape: (seq_len, 1)
position = torch.arange(0,seq_len).unsqueeze(1)  # shape: (seq_len, 1)
position

tensor([[0],
        [1],
        [2],
        [3],
        [4],
        [5],
        [6],
        [7],
        [8],
        [9]])

In [14]:
torch.arange(0, d_model, 2)

tensor([ 0,  2,  4,  6,  8, 10, 12, 14])

In [15]:
-math.log(10000.0) 

-9.210340371976184

In [13]:
div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))  # shape: (d_model/2,)

div_term

tensor([1.0000e+00, 3.1623e-01, 1.0000e-01, 3.1623e-02, 1.0000e-02, 3.1623e-03,
        1.0000e-03, 3.1623e-04])

### 3. mean and std in layer normalization class

In [16]:
import torch

# Simulate a batch of token embeddings
# batch_size=2, seq_len=3, d_model=4
x = torch.tensor([[[1.0, 2.0, 3.0, 4.0],   # sentence 1, token 1
                    [5.0, 6.0, 7.0, 8.0],   # sentence 1, token 2
                    [2.0, 2.0, 2.0, 2.0]],  # sentence 1, token 3
                   
                   [[1.0, 0.0, -1.0, 0.0],  # sentence 2, token 1
                    [3.0, 3.0,  3.0, 3.0],  # sentence 2, token 2
                    [1.0, 2.0,  3.0, 4.0]]]) # sentence 2, token 3

print("x shape:", x.shape)  # (2, 3, 4)

# dim=-1 means: compute mean/std across the last dimension (d_model)
# i.e. for each token, average its 4 features → collapses d_model to 1
mean = x.mean(dim=-1, keepdim=True)
std  = x.std(dim=-1, keepdim=True)

print("\nmean shape:", mean.shape)  # (2, 3, 1) — one mean per token
print("std shape: ", std.shape)    # (2, 3, 1) — one std  per token

print("\nmean:\n", mean)
# e.g. token [1,2,3,4] → mean = 2.5

print("\nstd:\n", std)
# e.g. token [1,2,3,4] → std ≈ 1.29

# keepdim=True keeps the shape (2,3,1) instead of (2,3)
# so that (x - mean) broadcasts correctly over the d_model dimension
print("\n(x - mean) shape:", (x - mean).shape)  # still (2, 3, 4) ✓

x shape: torch.Size([2, 3, 4])

mean shape: torch.Size([2, 3, 1])
std shape:  torch.Size([2, 3, 1])

mean:
 tensor([[[2.5000],
         [6.5000],
         [2.0000]],

        [[0.0000],
         [3.0000],
         [2.5000]]])

std:
 tensor([[[1.2910],
         [1.2910],
         [0.0000]],

        [[0.8165],
         [0.0000],
         [1.2910]]])

(x - mean) shape: torch.Size([2, 3, 4])


### 4. In Multiheadattention block

In [17]:
import torch

# Hyperparams
batch_size = 2
seq_len    = 5
d_model    = 512
h          = 8       # number of heads
d_k        = d_model // h  # 64

# Simulate query after W_q projection: (batch_size, seq_len, d_model)
query = torch.randn(batch_size, seq_len, d_model)
print("Before:", query.shape)  # (2, 5, 512)

# Step 1: .view() — split d_model into h heads of size d_k
query = query.view(batch_size, seq_len, h, d_k)
print("After view:", query.shape)  # (2, 5, 8, 64)
# Think of it as: each token's 512 features are split into 8 groups of 64

# Step 2: .transpose(1, 2) — swap seq_len and h dimensions
query = query.transpose(1, 2)
print("After transpose:", query.shape)  # (2, 8, 5, 64)
# Now shape is (batch_size, h, seq_len, d_k)
# i.e. head is the "outer" loop, seq_len is "inner"
# This lets each head independently attend over all tokens

# Why transpose? So attention can be computed per-head:
# For head 0: query[0, 0] has shape (5, 64) — all 5 tokens, head-0's 64 features
# For head 1: query[0, 1] has shape (5, 64) — all 5 tokens, head-1's 64 features
print("\nHead 0, sentence 0:", query[0, 0].shape)  # (5, 64)
print("Head 1, sentence 0:", query[0, 1].shape)  # (5, 64)

Before: torch.Size([2, 5, 512])
After view: torch.Size([2, 5, 8, 64])
After transpose: torch.Size([2, 8, 5, 64])

Head 0, sentence 0: torch.Size([5, 64])
Head 1, sentence 0: torch.Size([5, 64])


So the end result is identical to this conceptually:

what's actually happening under the hood, conceptually

head1_q = q @ W_q[:, 0:64]    # (batch, seq_len, 64)

head2_q = q @ W_q[:, 64:128]  # (batch, seq_len, 64)

head3_q = q @ W_q[:, 128:192] # (batch, seq_len, 64)

... and so on for all 8 heads

so there arent being 8 diff sets of Wq Wk Wv being created?

Correct — there is only one W_q, one W_k, one W_v each. But that single matrix is implicitly 8 sets packed together.
Think of it this way:
W_q is (512 × 512)
     = 8 blocks of (512 × 64) side by side

[ head1's W_q | head2's W_q | ... | head8's W_q ]

   (512×64)      (512×64)           (512×64)

Each head gets its own 64-column slice of W_q. Those slices are trained independently — they just live inside the same matrix for efficiency.